In [2]:
# use widgets as matplotlib backend
%matplotlib widget

# imports
from matplotlib import pyplot as plt
import os
import numpy
import os, sys
from time import time
from tqdm.auto import tqdm
from collections import defaultdict
import numpy as np
import matplotlib as mpl

# import the file object for opening kongsberg files
# Note: function and library naming to be discussed
from themachinethatgoesping.echosounders import kmall
from themachinethatgoesping.echosounders import index_functions
from themachinethatgoesping.pingprocessing import filter_pings

#simplify creating figures
mpl.rcParams['figure.dpi'] = 100
close_plots = True

def create_figure(name, return_ax=True):
    if close_plots: plt.close(name)
    fig = plt.figure(name)
    fig.suptitle = name

    if return_ax:
        return fig, fig.subplots()
    return fig

In [3]:
# list of folders with kongsberg .all or .wcd files (subfolders will be scanned as well)

folder = "../../../unittest_data/"

# list raw data files
files,index = index_functions.find_files_and_index(folder, [".kmall","kmwcd"])
files.sort()
            
files.sort()
file_name = files[0]
print("files:      ", len(files))

Found 22 files
files:       22


## Open all files
The function 
Notes: 
1. kmall.KMALLFileHandler(files) scanns and indexes all files and provides access to all files like a combined file stream
2. If a .all and a .wcd files with the same name (one .all and one .wcd) are added, they will be matched to a single file
3. It is not possible to add two .all or two .wcd with the same name, even if they are within different folders
4. Note: if the files are not sorted in time, the datagram packages will not be sorted by time either, however it isi simple to sort the pings at a later stage

In [4]:
# Index all files and initialize the file interfaces
# fm will be the accessor
fm = kmall.KMALLFileHandler(files)

#print some information about the files that where indexed
print(fm)

indexing files ⠐ 100% :00s<00m:00s] [..33285749584930.kmall (1/22)]                               
indexing files ⠠ 100% :00s<00m:00s] [..58516350552537.kmwcd (22/22)]                                
indexing files ⢀ 100% :00s<00m:00s] [Found: 502 datagrams in 22 files (11MB)]                                         
Initializing ping interface ⢀ 90% :00s<00m:00s] [Done]                                              
KMALLFileHandler
################
-
File infos 
-------------                 
- Number of loaded .kmall files: : 11       
- Number of loaded .kmwcd files: : 11       
- Total file size: :               11.28 MB 

 Detected datagrams 
^^^^^^^^^^^^^^^^^^^^ 
- timestamp_first:  26/07/2024 15:56:13.28 
- timestamp_last:   27/11/2025 14:14:45.23 
- Total:            502                    
- Datagrams [#MWC]: 58                     [M_WATER_COLUMN]
- Datagrams [#CHE]: 84                     [C_HEAVE]
- Datagrams [#SPE]: 33                     [S_POSITION_ERROR]
- Datagrams [#S

## How to access ping data
1. Pings describe the main highlevel interface of Ping
2. They are easy to access using fm.get_pings
3. Pings combine information from the interfaces (e.g. navigation or configuration) with raw datagrams that where produced for each ping
4. Pings implement simple functions to access difficult data (e.g get_sv) However, this is still work in progress
5. Pings also provide access to the raw data, this will be used for plotting later

In [5]:
# Extract the 11th single ping for test
ping = fm.get_pings()[11]

# print the georeferenced position and attitude of the transducer
print(ping.get_geolocation())

GeolocationLatLon (struct)
##########################
- latitude:  27°34'58.8"N  [ddd°mm',ss.s''N/S]
- longitude: 82°45'16.2"W  [ddd°mm',ss.s''E/W]
- z:         0.164         [positive downwards, m]
- yaw:       97.002        [90 ° at east]
- pitch:     1.861         [° positive bow up]
- roll:      4.097         [° positive port up]


In [6]:
# filter pings such that the for example must include water column and bottom detection data
pings = filter_pings.by_features(fm.get_pings(),['watercolumn','bottom'])
ping = pings[5]

In [7]:
# get access to the raw data of the specific ping
raw = ping.file_data

# print / access datagrams associated with the ping
print(raw.datagrams())

DatagramContainer
#################
-
Time info (Datagrams) 
------------------------ 
- Start time: 26/07/2024 15:56:14.18 
- End time:   26/07/2024 15:56:14.18 
- Sorted:     ascending              

 Contained datagrams 
--------------------- 
- Total:            2 
- Datagrams [#MWC]: 1 [M_WATER_COLUMN]
- Datagrams [#MRZ]: 1 [M_RANGE_AND_DEPTH]


In [9]:
#print the RawRangeAndAngle datagram associated with the ping
print(raw.datagrams("M_WATER_COLUMN")[0])

MWaterColumn
############
- bytes_datagram:      100948     [bytes]
- datagram_identifier: #MWC       [M_WATER_COLUMN]
- datagram_version:    2          
- system_id:           40         
- echo_sounder_id:     2040       
- time_sec:            1722009374 [s]
- time_nanosec:        178447080  [ns]

 date/time 
-----------  
- timestamp: 1722.009e⁶   [s]
- date:      26/07/2024   [MM/DD/YYYY]
- time:      15:56:14.178 [HH:MM:SS]

 MPartition part 
----------------- 
- number_of_datagrams: 1 
- datagram_number:     1 

 datagram content 
------------------ 
- bytes_content:            12    
- ping_count:               64052 
- rx_fans_per_ping:         1     
- rx_fan_index:             0     
- swaths_per_ping:          1     
- swath_along_position:     0     
- tx_transducer_ind:        0     
- rx_transducer_ind:        0     
- number_of_rx_transducers: 1     
- algorithm_type:           0     

 Tx info (.tx_info) 
-------------------- 
- bytes_content:           12            [

In [19]:
# read_merged_watercolumndatagram() holds a variable that is called skip_data
# if the true, the sample data will be skipped
# in this case you cannot access sample data, but looping through the data is accelerated 
# This is usefull to extract e.g statistical information in a first loop
# Speedup is not very impressive though (typically factor 2) because of the complicated datagram structure of .wcd files

# check if there where different tvgs applied in the datasets
tvgs = defaultdict(int)
for i,p in enumerate(tqdm(pings)):
    try:    
        w = p.file_data.datagrams("M_WATER_COLUMN", skip_data = True)[0]
              
    except IndexError as e:
        print("error:",i,e,"|",type(e))
    except ValueError as e:
        print("error:",i,e,"|",type(e))
    except RuntimeError as e:
        print("error:",i,e,"|",type(e))


  0%|          | 0/40 [00:00<?, ?it/s]